# Evo1 DNA Sequence Generation and Scoring

![Evo1 DNA Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/evo1/hero.png)

This notebook demonstrates both tools that utilize Evo1 model. In the first section, we use `run_evo1_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling. In the second section, we use `run_evo1_score` to compute log-likelihood and perplexity metrics for a set of DNA sequences, which can be used to assess sequence quality or compare design candidates.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo1")
display_overview("evo1")
display_docs_section("evo1", "Background")

# Evo1

Evo1 is a 7-billion parameter DNA language model built on the [StripedHyena](https://en.wikipedia.org/wiki/State_space_model#Deep_learning) architecture, trained on 2.7 million [prokaryotic](https://en.wikipedia.org/wiki/Prokaryote) and [phage](https://en.wikipedia.org/wiki/Bacteriophage) genomes from the OpenGenome dataset. This tool performs autoregressive DNA sequence generation from prompts and optionally scores generated sequences by mean log-probability.

## Background

**What does this tool measure/predict?**
Evo1 generates novel DNA sequences by learning the statistical patterns of prokaryotic and phage genomes. It can also score sequences by log-likelihood, providing a measure of how "natural" a generated sequence appears relative to the training distribution.

**Why is this important?**
Generative DNA models enable de novo design of biological sequences; genes, [operons](https://en.wikipedia.org/wiki/Operon), [CRISPR](https://en.wikipedia.org/wiki/CRISPR) systems, and [mobile genetic elements](https://en.wikipedia.org/wiki/Transposable_element); without requiring template sequences. By learning the grammar of genomes at single-nucleotide resolution, Evo1 can propose functional DNA that respects codon usage, regulatory signals, and structural constraints.

**Scientific foundation:**
Evo1 uses the StripedHyena architecture, a hybrid state-space/attention model that processes DNA at single-nucleotide resolution. Unlike transformer-only models, StripedHyena supports efficient long-range sequence modeling up to 131k tokens. The model is trained with a standard autoregressive (next-token prediction) objective on raw genomic DNA.

## Available tools

In [2]:
display_available_tools("evo1")

- **`run_evo1_sample()`** — Sample DNA sequences using Evo1 language model
- **`run_evo1_score()`** — Score DNA sequences using Evo1 language model

### `run_evo1_sample`

Evo1 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over prokaryotic and phage genomes. The `temperature` and `top_k` parameters control the diversity of the output: lower values produce more conservative, high-confidence extensions while higher values allow more exploratory generation. Fine-tuned checkpoints are also available for CRISPR and transposon-specific generation.

In [3]:
from proto_tools import (
    Evo1SampleInput,
    Evo1SampleConfig,
    run_evo1_sample,
)

In [4]:
# Dipslay docs
display_api_reference("evo1", "input", "run_evo1_sample")

# Create a simple Evo1 Input with one starting prompt
inputs = Evo1SampleInput(prompts=["ACGTACGT"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `list[str]` | required | Prompt sequence(s) to condition generation on |

In [5]:
# Display config docs
display_api_reference("evo1", "config", "run_evo1_sample")

# Create a simple Evo1Sample config | see docs below for all fields
config = Evo1SampleConfig(
    model_name="evo-1-8k-base",
    max_new_tokens=200,
    temperature=0.1,
    top_k=4,
    prepend_prompt=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `Evo1SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `1800` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `prepend_prompt` | `bool` | `False` | Include the input prompt at the start of each generated sequence |
| `temperature` | `float` | `1.0` | Softmax temperature for sampling; lower is more deterministic |
| `top_p` | `float` | `1.0` | Nucleus sampling cutoff over per-position token probabilities |
| `batch_size` | `int` | `1` | Prompts per GPU forward pass; raise for throughput, lower if OOM |
| `model_name` | `Literal['evo-1.5-8k-base', 'evo-1-8k-base', 'evo-1-131k-base', 'evo-1-8k-crispr', 'evo-1-8k-transposon']` | `'evo-1-8k-base'` | Evo1 weights variant |
| `top_k` | `int` | `4` | Limit sampling to the top-k most probable tokens at each step |
| `max_new_tokens` | `int` | `100` | Maximum number of new tokens to generate per prompt |
| `cached_generation` | `bool` | `True` | Use the KV cache for autoregressive generation |
| `force_prompt_threshold` | `int` | `128` | Tokens to prefill in parallel before switching to prompt forcing |

In [6]:
# Run the sampling function
sample_result = run_evo1_sample(inputs, config)

Running run_evo1_sample [00:00]

In [7]:
display_api_reference("evo1", "output", "run_evo1_sample")

# Select the prompt and corresponding generated output sequence and show some results
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")
if sample_result.scores:
    score = sample_result.scores[0]
    print(f"Avg log-likelihood: {score['avg_log_likelihood']:.4f}")
    print(f"Perplexity:         {score['perplexity']:.4f}")

**Output** — `Evo1SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Generated sequences |
| `scores` | `list[proto_tools.tools.causal_models.shared_data_models.CausalModelScoringMetrics] \| None` | `None` | Scoring metrics per generated sequence |

Prompt:       ACGTACGT
Generated:    ACGTACGTGAAGCCGACGTCGTGGCGGGTGCGCGCGTACTCGGCGAGGTCGTCGAGCACGAGCGCGCCCTCGAGCACGAT...
Total length: 208 nucleotides
Avg log-likelihood: -2.7608
Perplexity:         15.8125


### `run_evo1_score`

Evo1 scores sequences by computing the autoregressive log-likelihood at each position, measuring how well each nucleotide follows from its preceding context according to the model. The resulting perplexity is a single number summarizing sequence naturalness: lower perplexity indicates that the sequence more closely matches the statistical patterns of prokaryotic and phage genomes in the training data. This is useful for ranking generated candidates, filtering out low-quality designs, or comparing the naturalness of wild-type versus engineered sequences.

In [8]:
from proto_tools import (
    Evo1ScoringInput,
    Evo1ScoringConfig,
    run_evo1_score,
)

In [9]:
# Display docs
display_api_reference("evo1", "input", "run_evo1_score")

# Score two short DNA sequences
score_inputs = Evo1ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Sequences to score |

In [10]:
# Display config docs
display_api_reference("evo1", "config", "run_evo1_score")

# Create a simple Evo1Scoring config | see docs above for all fields
score_config = Evo1ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `Evo1ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `1800` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `model_name` | `Literal['evo-1.5-8k-base', 'evo-1-8k-base', 'evo-1-131k-base', 'evo-1-8k-crispr', 'evo-1-8k-transposon']` | `'evo-1-8k-base'` | Evo1 weights variant |

In [11]:
# Run the scoring function
score_result = run_evo1_score(score_inputs, score_config)

Running run_evo1_score [00:00]

In [12]:
# Display output docs
display_api_reference("evo1", "output", "run_evo1_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.causal_models.shared_data_models.CausalModelScoringMetrics]` | required | List of scoring outputs, one per input sequence |

Sequence 1: ATGCGTAAA
    Log-likelihood:     -13.398
    Avg log-likelihood: -1.489
    Perplexity:         4.431
Sequence 2: ATGCGTAAATTT
    Log-likelihood:     -17.062
    Avg log-likelihood: -1.422
    Perplexity:         4.145


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo1_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo1_sequences.fasta'}")

score_result.export(name="evo1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo1_scores.csv'}")

Generated sequences exported to example_output/evo1_sequences.fasta
Scores exported to example_output/evo1_scores.csv
